# 10 NumPy operations

Element-wise maths, summaries along an axis, broadcasting, missing values and a little linear algebra.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
# 5 patients x 3 tests: hb, wbc, crp
labs = np.column_stack([
    rng.normal(12.5, 1.5, 5),
    rng.normal(7, 2, 5),
    rng.lognormal(1.5, 1, 5),
]).round(1)
labs

## Element-wise maths

In [ ]:
print(np.sqrt(labs[:, 2]).round(2))
print(np.log(labs[:, 2]).round(2))
print(np.round(labs * 1.1, 1))

## Summaries and axes

`axis=0` works down the rows (one result per column).
`axis=1` works across the columns (one result per row).

In [ ]:
print("column means:", labs.mean(axis=0).round(2))
print("row max:     ", labs.max(axis=1))
print("overall sd:  ", labs.std().round(2))
print(np.percentile(labs[:, 0], [25, 50, 75]).round(2))

## Broadcasting

When shapes differ, NumPy stretches the smaller one where it can.
Standardising each column (z-scores) is one line:

In [ ]:
z = (labs - labs.mean(axis=0)) / labs.std(axis=0)
print(z.round(2))
print(z.mean(axis=0).round(10), z.std(axis=0))

Scaling like this is exactly what `StandardScaler` does in scikit-learn.

## Missing values

`np.nan` marks a missing number. Ordinary functions return `nan` if any value is missing;
the `nan*` versions skip them.

In [ ]:
x = np.array([12.1, np.nan, 13.8, 9.7])
print(x.mean(), np.nanmean(x))
print(np.isnan(x), np.isnan(x).sum())
print(np.nan == np.nan)      # nan is not equal to itself: always use isnan

## where, clip, unique, sort

In [ ]:
hb = np.array([12.1, 10.4, 13.8, 9.7, 11.2])
print(np.where(hb < 11, "Low", "Normal"))
print(np.clip(hb, 10, 13))
print(np.unique(["S", "R", "S", "I", "R"], return_counts=True))
print(np.sort(hb), np.argsort(hb))

## Combining arrays

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6]])
print(np.vstack([a, b]))
print(np.hstack([a, a]))
print(np.concatenate([a, b], axis=0))

## Linear algebra

Machine learning models are mostly matrix maths. A linear model predicts `X @ w + b`.

In [ ]:
X = np.array([[1.0, 2.0], [2.0, 0.5], [3.0, 1.5]])
w = np.array([0.4, -0.2])
print(X @ w)                         # matrix times vector
print(X.T @ X)                       # transpose times itself

# least squares fit of y = X w, the core of linear regression
y = np.array([1.1, 0.6, 1.0])
w_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
print(w_hat.round(3))

## Why vectorise

Loops in Python are slow for large data. Compare:

In [ ]:
big = rng.normal(size=1_000_000)

import time
t = time.perf_counter()
total = 0.0
for v in big:
    total += v * v
loop_time = time.perf_counter() - t

t = time.perf_counter()
total2 = (big * big).sum()
vec_time = time.perf_counter() - t

print(f"loop {loop_time:.3f}s, numpy {vec_time:.4f}s, about {loop_time / vec_time:.0f}x faster")